# Lab 10 - LLMs

### Install Libraries

In [5]:
!pip install -q transformers sentencepiece

### Import Libraries

In [2]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForQuestionAnswering
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: Tesla T4


### Load Models

In [3]:
# Summarizer
summarizer_tokenizer = AutoTokenizer.from_pretrained(
    "facebook/bart-large-cnn"
)

summarizer_model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/bart-large-cnn"
).to(device)

summarizer_model.eval()



# QA
qa_tokenizer = AutoTokenizer.from_pretrained(
    "distilbert/distilbert-base-cased-distilled-squad"
)

qa_model = AutoModelForQuestionAnswering.from_pretrained(
    "distilbert/distilbert-base-cased-distilled-squad"
).to(device)

qa_model.eval()

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

... (7 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  261MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DistilBertForQuestionAnswering(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
     

### Text Summarization

In [4]:
paragraph = """
Artificial Intelligence (AI) is becoming an essential part of modern society, transforming the way people live, work, and communicate.
AI allows computers and machines to perform tasks that normally require human intelligence, such as recognizing images, understanding languages, solving problems, and making decisions.
Machine learning enables systems to learn from data and improve their performance over time, while deep learning uses complex neural networks to solve more advanced tasks.
Today, AI is applied in many industries, including healthcare, education, banking, transportation, and entertainment.
In healthcare, AI helps doctors analyze medical images, detect diseases, and recommend suitable treatments.
In education, AI-powered tools provide personalized learning experiences and support students with their studies.
Businesses also use AI to automate repetitive tasks, analyze customer behavior, and improve productivity.
Furthermore, AI plays an important role in developing self-driving vehicles, virtual assistants, recommendation systems, and intelligent robots.
Despite these advantages, the development of AI also creates several challenges.
Privacy and data security are major concerns because AI systems often require large amounts of personal information.
Additionally, automation may replace certain jobs, requiring workers to develop new skills to adapt to changing industries.
Another important issue is the possibility of biased decisions caused by inaccurate or unbalanced training data.
Therefore, AI should be developed and used responsibly, with appropriate regulations and human supervision. Governments, organizations, and researchers must work together to ensure that AI technologies remain safe, transparent, and beneficial to society.
Overall, AI has significant potential to improve efficiency, solve complex problems, and create new opportunities.
However, balancing technological progress with ethical considerations is necessary to ensure a positive future for everyone.
"""

print(paragraph)

inputs = summarizer_tokenizer(
    paragraph,
    max_length=1024,
    truncation=True,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    summary_ids = summarizer_model.generate(
        inputs["input_ids"],
        max_length=100,
        min_length=30,
        num_beams=4,
        do_sample=False
    )

summary = summarizer_tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("\nOriginal words:", len(paragraph.split()))
print("Summary words:", len(summary.split()))

print("\nGenerated Summary:\n")
print(summary)


Artificial Intelligence (AI) is becoming an essential part of modern society, transforming the way people live, work, and communicate. 
AI allows computers and machines to perform tasks that normally require human intelligence, such as recognizing images, understanding languages, solving problems, and making decisions. 
Machine learning enables systems to learn from data and improve their performance over time, while deep learning uses complex neural networks to solve more advanced tasks. 
Today, AI is applied in many industries, including healthcare, education, banking, transportation, and entertainment. 
In healthcare, AI helps doctors analyze medical images, detect diseases, and recommend suitable treatments. 
In education, AI-powered tools provide personalized learning experiences and support students with their studies. 
Businesses also use AI to automate repetitive tasks, analyze customer behavior, and improve productivity. 
Furthermore, AI plays an important role in developing 

### Question Answering

1. Single Question

In [5]:
question = "What is a subset of AI?"

inputs = qa_tokenizer(
    question,
    paragraph,
    return_tensors="pt",
    truncation=True
).to(device)

with torch.no_grad():
    outputs = qa_model(**inputs)

answer_start = torch.argmax(outputs.start_logits)
answer_end = torch.argmax(outputs.end_logits) + 1

answer_tokens = inputs["input_ids"][0][answer_start:answer_end]

answer = qa_tokenizer.decode(
    answer_tokens,
    skip_special_tokens=True
)

print("Question:", question)
print("Answer:", answer)

Question: What is a subset of AI?
Answer: Artificial Intelligence


2. Multiple Question

In [9]:
questions = [
    "What is the definition of AI?",
    "What is machine learning?",
    "Where is AI commonly used?",
    "What are the concerns about AI?"
]

for question in questions:

    inputs = qa_tokenizer(
        question,
        paragraph,
        return_tensors="pt",
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = qa_model(**inputs)

    answer_start = torch.argmax(outputs.start_logits)
    answer_end = torch.argmax(outputs.end_logits) + 1

    answer_tokens = inputs["input_ids"][0][answer_start:answer_end]

    answer = qa_tokenizer.decode(
        answer_tokens,
        skip_special_tokens=True
    )

    print("\nQuestion:", question)
    print("Answer:", answer ,"\n")


Question: What is the definition of AI?
Answer: Artificial Intelligence 


Question: What is machine learning?
Answer: enables systems to learn from data and improve their performance over time 


Question: Where is AI commonly used?
Answer: healthcare, education, banking, transportation, and entertainment 


Question: What are the concerns about AI?
Answer: Privacy and data security 



3. Interactive Question

In [10]:
print("Ask questions based on the paragraph")
print("Type 'quit' to exit")

while True:

    question = input("\nEnter your question: ")

    if question.lower() == "quit":
        print("Terminated")
        break

    if question.strip() == "":
        print("Please enter a valid question")
        continue

    inputs = qa_tokenizer(
        question,
        paragraph,
        return_tensors="pt",
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = qa_model(**inputs)

    answer_start = torch.argmax(outputs.start_logits)
    answer_end = torch.argmax(outputs.end_logits) + 1

    answer_tokens = inputs["input_ids"][0][answer_start:answer_end]

    answer = qa_tokenizer.decode(
        answer_tokens,
        skip_special_tokens=True
    )

    print("\nAnswer:", answer)

Ask questions based on the paragraph
Type 'quit' to exit

Answer: allows computers and machines to perform tasks that normally require human intelligence

Answer: allows computers and machines to perform tasks that normally require human intelligence

Answer: AI - powered tools provide personalized learning experiences and support students with their studies

Answer: helps doctors analyze medical images, detect diseases, and recommend suitable treatments

Answer: allows computers and machines to perform tasks that normally require human intelligence

Answer: 

Answer: 

Enter your question: quit
Terminated
